# 03 · Extreme Value Analysis and IDF Curves

**Author:** Salvador Navas  
**Basin:** Río Besaya — Cantabria

Starting from annual daily rainfall maxima, this notebook:
- Fits GEV and Gumbel distributions to annual block maxima (BM)
- Estimates return levels for T = 2, 5, 10, 25, 50, 100, 500 years
- Constructs IDF curves for the basin using duration-scaling relationships
- Exports parameters and quantiles for Notebook 04

**Methodological note.** The record has 43 annual maxima. Return levels beyond
about 100 years are extrapolations with high uncertainty; the 500-year value is
kept for workflow continuity, not as a calibrated design recommendation.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pyhydra.climate.time_series.extremes import (
    extract_block_maxima,
    fit_gev, fit_gpd,
    return_levels,
    return_level_ci,
    plot_return_levels,
    plot_diagnostic,
)

# ── Rutas ──────────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR  = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))
DATA_ROOT = DATA_DIR / 'pilot_cases' / 'los_corrales_buelna'
PROC_DIR  = DATA_ROOT / 'processed'

RETURN_PERIODS = [2, 5, 10, 25, 50, 100, 500]
DURATIONS_H    = [1, 2, 3, 6, 12, 24]   # horas — 24h = datos diarios

---
## 1. Load daily AMR and extract annual maxima

The annual maximum series (AMS) is extracted from daily AMR at the "all-basin" 
scale — i.e., the spatial average over the entire Besaya catchment.

**Why use AMS and not the raw daily series?**  
The GEV Extreme Value Theorem guarantees that block maxima (one per year) converge 
to a GEV distribution as block size grows, regardless of the daily distribution. 
With 43 years of data, the AMS provides 43 near-independent data points for fitting.

Two data sources are supported:
1. `.dat` files from AEMET/SAIH (pre-computed per-station annual maxima)
2. Daily AMR matrix from Notebook 02 — extract via `resample('YE').max()`

The notebook uses Option A if `.dat` files exist, otherwise falls back to Option B.


In [2]:
# ── Option A: load pre-computed annual maxima (.dat per station) ────────────
# DAT_FILES = sorted((DATA_ROOT / 'gauges' / 'rainfall').glob('Maximos_anual_*.dat'))
# print(f'Ficheros .dat encontrados: {[f.stem for f in DAT_FILES]}')
# 
# ann_max_series = {}
# for dat in DAT_FILES:
#     station = dat.stem.replace('Maximos_anual_', '')
#     values  = np.loadtxt(dat)          # columna simple de valores en mm/día
#     n       = len(values)
#     # Assign years (last year of record is approx. 2012)
#     years   = range(2012 - n + 1, 2013)
#     idx     = pd.to_datetime([f'{y}-12-31' for y in years])
#     ann_max_series[station] = pd.Series(values, index=idx, name=station)
# 
# ann_max_df = pd.DataFrame(ann_max_series)
# print(f'Máximos anuales por estación: {ann_max_df.shape}')
# print(ann_max_df.describe().round(1))
# 
# # Spatial mean for basin frequency analysis
# annual_max = ann_max_df.mean(axis=1).dropna()
# annual_max.name = 'PMA_Besaya'

# ── Option B: compute from interpolated daily AMR (requires Notebook 02) ───
pma = pd.read_csv(PROC_DIR / 'pma_idw_daily.csv', index_col=0, parse_dates=True)
annual_max = extract_block_maxima(pma.mean(axis=1), freq='YE').dropna()
annual_max.name = 'PMA_Besaya'
# ──────────────────────────────────────────────────────────────────────────────

print(f'\nMáximos anuales de la cuenca: {len(annual_max)} años')
print(f'  min={annual_max.min():.1f}  media={annual_max.mean():.1f}  max={annual_max.max():.1f} mm/día')
annual_max.tail()



Máximos anuales de la cuenca: 43 años
  min=30.9  media=46.7  max=73.6 mm/día


---
## 2. GEV and Gumbel fitting

**GEV vs Gumbel — which to use?**

| Model | Parameters | Shape ξ | Tail |
|-------|-----------|---------|------|
| **GEV** | μ, σ, ξ (3 params) | Free | Flexible (Fréchet/Gumbel/Weibull) |
| **Gumbel** | μ, σ (2 params) | Fixed = 0 | Light exponential tail |

For Atlantic-climate precipitation, ξ is typically positive (Fréchet family, 
heavier tail than exponential). Fixing ξ = 0 (Gumbel) reduces the number of 
parameters but may underestimate extreme return levels if ξ > 0 in reality.

**Fitting method — MLE with bias correction:**  
With n = 43, MLE is reliable for GEV (asymptotic properties kick in at n ≥ 30). 
L-moments are used as a warm-start for MLE (already done internally by `fit_gev`).

**Reading the diagnostic panel:**
- PP plot: empirical vs theoretical cumulative probability — points should be on the 1:1 diagonal
- QQ plot: empirical vs theoretical quantiles — curvature indicates ξ is wrong
- Return level: data points should fall within the 95% CI band
- Density: histogram should match the fitted PDF


In [3]:
# GEV (MLE with bias correction)
params_gev = fit_gev(annual_max.values, method='mle')
print('GEV    — mu={mu:.2f}  sigma={sigma:.2f}  xi={xi:.3f}'.format(**params_gev))

# Gumbel = GEV special case with xi = 0 (fitted via scipy gumbel_r)
from scipy.stats import gumbel_r as _gumbel
_loc, _sc = _gumbel.fit(annual_max.values)
params_gumbel = {'mu': float(_loc), 'sigma': float(_sc), 'xi': 0.0}
print('Gumbel — mu={mu:.2f}  sigma={sigma:.2f}  xi=0 (fixed)'.format(**params_gumbel))


GEV    — mu=42.22  sigma=8.38  xi=-0.049
Gumbel — mu=42.01  sigma=8.24  xi=0 (fixed)


In [4]:
# Visual diagnostic
fig = plot_diagnostic(annual_max.values, params_gev, dist='gev')
plt.savefig(PROC_DIR / 'gev_diagnostic.png', dpi=150)
plt.show()


---
## 3. Return levels and confidence bands (95 %)

The 95% bootstrap confidence interval is computed by resampling the AMS with 
replacement 500 times and refitting the GEV at each resample.

**Interpreting the results:**

| T (years) | Interpretation | Note |
|-----------|---------------|------|
| 2 | Median flood year — exceeded every other year on average | Well-constrained by data |
| 10 | Design standard for secondary drainage and minor bridges | Reliable (within record) |
| 25 | Typical urban drainage design standard | At the limit of the record |
| 100 | Primary infrastructure, flood hazard mapping | Extrapolation — CI widens significantly |
| 500 | Extreme scenario, dam safety, critical infrastructure | High uncertainty; CI may be 3–5× the point estimate |

> **Rule of thumb:** Do not use extrapolated return levels (T > 2× record length) 
> without comparing against regional estimates from neighbouring basins.
> The Cantabrian basin network (CHC) publishes regional T-year quantiles.


In [5]:
T_plot = np.logspace(np.log10(1.1), np.log10(1000), 200)

rl_gev    = return_levels(params_gev,    T_plot, dist='gev')
rl_gumbel = return_levels(params_gumbel, T_plot, dist='gev')

# Bootstrap confidence intervals — one call per return period
ci_gev = {}
for T in RETURN_PERIODS:
    ci_gev[T] = return_level_ci(annual_max.values, T,
                                 dist='gev', n_bootstrap=200, ci=0.95)
# ci_gev[T] = (estimate, lower_ci, upper_ci)

fig = plot_return_levels(
    annual_max.values, params_gev,
    dist='gev', ci=0.95
)
plt.savefig(PROC_DIR / 'return_levels.png', dpi=150)
plt.show()


In [6]:
# Quantile table for T = RETURN_PERIODS
rl_table = pd.DataFrame({
    'T_years':   RETURN_PERIODS,
    'GEV_mm':    [return_levels(params_gev,    [T], dist='gev').iloc[0]
                  for T in RETURN_PERIODS],
    'Gumbel_mm': [return_levels(params_gumbel, [T], dist='gev').iloc[0]
                  for T in RETURN_PERIODS],
    'CI_low_mm':  [ci_gev[T][1] for T in RETURN_PERIODS],
    'CI_high_mm': [ci_gev[T][2] for T in RETURN_PERIODS],
})
rl_table = rl_table.round(1)
print(rl_table.to_string(index=False))
rl_table.to_csv(PROC_DIR / 'return_levels_24h.csv', index=False)


 T_years  GEV_mm  Gumbel_mm  CI_low_mm  CI_high_mm
       2    45.3       45.0       41.3        48.4
       5    54.4       54.4       50.1        57.7
      10    60.1       60.5       54.5        63.9
      25    67.1       68.4       59.3        74.1
      50    72.0       74.2       60.9        82.6
     100    76.8       79.9       62.5        91.9
     500    87.2       93.2       65.2       119.9


---
## 4. IDF curves (Intensity–Duration–Frequency)

IDF curves convert the 24-hour design depth P₂₄ₕ(T) to sub-daily intensities 
for each duration d (1h, 2h, 3h, 6h, 12h, 24h). This is needed because HEC-HMS 
requires a **time-distributed hyetograph**, not just a total depth.

### Témez scaling relationship

The Témez method (MOPU, 1992) is the standard in Spanish hydrology:
$$P_d = P_{24h} \cdot \left(rac{d}{24}ight)^{0.28}$$
$$I_d = P_d / d \quad (	ext{mm/h})$$

The exponent **0.28** is calibrated for the Atlantic/Cantabrian climate zone. Other 
Spanish climate zones use different exponents:
| Climate zone | Exponent | Example |
|-------------|---------|---------|
| Atlantic (Cantabria, Galicia) | 0.28 | This study |
| Mediterranean (Valencia, SE Spain) | 0.32–0.35 | More intense sub-daily |
| Continental (Meseta) | 0.25–0.28 | Intermediate |

> **Limitation:** The Témez relationship is a statistical average fitted to a network 
> of rain gauge records. For basins with significant elevation variation, the exponent 
> may vary between high-altitude storm cells (more intense) and coastal showers.
> Use hourly rain gauge data to calibrate station-specific exponents if available.


In [7]:
TEMEZ_EXPONENT = 0.28   # exponent for Cantabria / Atlantic climate

idf_records = []
for T in RETURN_PERIODS:
    p24 = return_levels(params_gev, [T], dist='gev').iloc[0]
    for d in DURATIONS_H:
        pd_mm  = p24 * (d / 24) ** (1 - TEMEZ_EXPONENT)
        i_mmh  = pd_mm / d
        idf_records.append({'T_years': T, 'duration_h': d,
                            'depth_mm': round(pd_mm, 1),
                            'intensity_mmh': round(i_mmh, 2)})

idf = pd.DataFrame(idf_records)
idf_pivot = idf.pivot(index='duration_h', columns='T_years', values='depth_mm')
print('IDF — Profundidades de lluvia (mm):')
print(idf_pivot.to_string())
idf.to_csv(PROC_DIR / 'idf_curves.csv', index=False)


IDF — Profundidades de lluvia (mm):
T_years      2     5     10    25    50    100   500
duration_h                                          
1            4.6   5.5   6.1   6.8   7.3   7.8   8.8
2            7.6   9.1  10.0  11.2  12.0  12.8  14.6
3           10.1  12.2  13.4  15.0  16.1  17.2  19.5
6           16.7  20.0  22.1  24.7  26.5  28.3  32.1
12          27.5  33.0  36.5  40.7  43.7  46.6  52.9
24          45.3  54.4  60.1  67.1  72.0  76.8  87.2


In [8]:
idf_int = idf.pivot(index='duration_h', columns='T_years', values='intensity_mmh')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.plasma(np.linspace(0, 0.85, len(RETURN_PERIODS)))
for (T, col), c in zip(idf_pivot.items(), colors):
    axes[0].plot(idf_pivot.index, col, 'o-', label=f'T={T} años', color=c)
axes[0].set(xlabel='Duración (h)', ylabel='Profundidad (mm)',
            title='IDF — Profundidad', xscale='log', yscale='log')
axes[0].legend(fontsize=8)
axes[0].grid(True, which='both', alpha=0.3)

for (T, col), c in zip(idf_int.items(), colors):
    axes[1].plot(idf_int.index, col, 'o-', label=f'T={T} años', color=c)
axes[1].set(xlabel='Duración (h)', ylabel='Intensidad (mm/h)',
            title='IDF — Intensidad', xscale='log', yscale='log')
axes[1].legend(fontsize=8)
axes[1].grid(True, which='both', alpha=0.3)

plt.suptitle('Curvas IDF — Cuenca del Besaya', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROC_DIR / 'idf_curves.png', dpi=150)
plt.show()

---
## 5. Seasonal analysis of extreme events

Understanding **which months produce the largest rainfall maxima** is important for:
- Selecting the design storm season (IDF curves may be season-dependent)
- Choosing the antecedent soil moisture condition for HMS simulation
- Validating the homogeneity of the AMS (mixing summer and winter maxima 
  can violate the GEV i.i.d. assumption)

For the Besaya (Cantabrian climate), most annual maxima occur in October–January 
(Atlantic frontal systems). Summer convective maxima are rarer but shorter and more 
intense — they may produce different flood hydrograph shapes (faster rising limb).

> If > 20% of maxima occur in a different season (e.g., summer), consider 
> seasonal GEV fitting or a 2-component mixture model.


In [9]:
# Month in which each annual maximum occurs
annual_max_df = pd.DataFrame({'pma_mm': annual_max.values}, index=annual_max.index)
annual_max_df['month'] = annual_max_df.index.month

monthly_counts = annual_max_df['month'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(monthly_counts.index,  monthly_counts.values, color='steelblue', edgecolor='k')
ax.set(xlabel='Mes', ylabel='Frecuencia de máximos anuales',
       title='Estacionalidad de los máximos anuales — PMA Besaya',
       xticks=range(1, 13),
       xticklabels=['Ene','Feb','Mar','Abr','May','Jun',
                    'Jul','Ago','Sep','Oct','Nov','Dic'])
plt.tight_layout()
plt.savefig(PROC_DIR / 'seasonality_maxima.png', dpi=150)
plt.show()

---
## 6. Summary of exported parameters

The JSON file `extreme_value_params.json` stores:

```json
{
  "n_years": 43,
  "period_start": "...",  "period_end": "...",
  "gev_mu": ..., "gev_sigma": ..., "gev_xi": ...,
  "gumbel_mu": ..., "gumbel_sigma": ...,
  "return_levels": {"T10": ..., "T25": ..., "T100": ...},
  "idf_temez_exponent": 0.28,
  "idf": {  "T25": {"1h": ..., "6h": ..., "24h": ...}, ... }
}
```

Used directly by:
- **Notebook 04** → build alternating-block hyetographs from IDF table
- **Notebook 08** → apply CC delta factors to the T-year design depths


In [10]:
import json

summary = {
    'n_years':      int(len(annual_max)),
    'period_start': str(annual_max.index[0].date()),
    'period_end':   str(annual_max.index[-1].date()),
    'gev_params':   {k: round(float(v), 4) for k, v in params_gev.items()},
    'gumbel_params': {k: round(float(v), 4) for k, v in params_gumbel.items()},
    'return_levels_24h': {str(T): round(float(return_levels(params_gev, [T], dist='gev').iloc[0]), 1)
                          for T in RETURN_PERIODS},
    'idf_temez_exponent': TEMEZ_EXPONENT,
}

with open(PROC_DIR / 'extreme_value_params.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Exportado: extreme_value_params.json')
print(json.dumps(summary, indent=2))

Exportado: extreme_value_params.json
{
  "n_years": 43,
  "period_start": "1970-12-31",
  "period_end": "2012-12-31",
  "gev_params": {
    "mu": 42.2223,
    "sigma": 8.3834,
    "xi": -0.0485
  },
  "gumbel_params": {
    "mu": 42.0053,
    "sigma": 8.2388,
    "xi": 0.0
  },
  "return_levels_24h": {
    "2": 45.3,
    "5": 54.4,
    "10": 60.1,
    "25": 67.1,
    "50": 72.0,
    "100": 76.8,
    "500": 87.2
  },
  "idf_temez_exponent": 0.28
}
